# Man-in-the-Middle — Blackout

The attacker interposed on the GCS<->vehicle link **blinds the GCS**:

- **Downlink:** all telemetry to the GCS is dropped (position, mission state,
  attitude). The attacker keeps `HEARTBEAT` alive so the link still *looks*
  healthy, and lets the `LOGIC_DONE` completion signal through so the sim can
  terminate.
- **Uplink:** every command from the GCS is dropped — the operator cannot
  reach the vehicle.

The GCS still tries its usual intervention (redirect east), but because it
never sees `MISSION_CURRENT`, the trigger never fires — and even if it did, the
command would be suppressed. The drone therefore flies its **full north
mission unbothered** while the GCS sees nothing. Gazebo still shows the true
flight (the Oracle/visualizer path is independent of the GCS link).

In [ ]:
from simulator import Simulator
from simulator.config import PARAMS_PATH, Color, Model
from simulator.entities import SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin and waypoints

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, 0, 0, 0)
speed = 5.0    # m/s
cruise_alt = 10.0  # m
model = Model.IRIS
sysid = 1

# Mission seq: seq=0 home, seq=1 TAKEOFF, seq=2 flying to north_100,
#              seq=3 flying to north_200
home_wp    = ENU(x=0, y=0,   z=0)
north_100  = ENU(x=0, y=100, z=cruise_alt)
north_200  = ENU(x=0, y=200, z=cruise_alt)
mission_wps = [home_wp, north_100, north_200]

# The GCS will *try* to redirect east — the attacker blocks it.
east_target = gra_origin.unpose().to_abs(ENU(x=100, y=0, z=cruise_alt))
print(f"GCS would redirect to: lat={east_target.lat:.7f}, lon={east_target.lon:.7f}")

## Vehicle

In [ ]:
mission_path = "simulator/planner/missions/mitm_north.waypoints"

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=mission_path,
    navigation_speed=speed,
    firmware=model.firmware,
)

vehicle = SimVehicle.from_relative(
    sysid=sysid,
    gcs_name=f"BLUE_{Color.BLUE.emoji}",
    plan=plan,
    color=Color.BLUE,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    model=model,
)

## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
gaz.markers.append(origin_marker)

## Simulator + MITM blackout

In [ ]:
simulator = Simulator(visualizer=gaz, verbose=1)
simulator.add_vehicle(vehicle, parm=str(PARAMS_PATH / "vehicle.parm"))

# The GCS still attempts its redirect — it just never reaches the vehicle.
# `simulator.intervention` is keyed by sysid, so this targets only this vehicle.
simulator.intervention[sysid] = {
    "trigger_seq": 3,
    "target_lat": east_target.lat,
    "target_lon": east_target.lon,
    "target_alt": cruise_alt,
}

# Attacker blinds the GCS for this vehicle only: drop all telemetry (except
# heartbeat + LOGIC_DONE) and all commands. `simulator.mitm` is keyed by
# sysid — other vehicles sharing this GCS would be unaffected.
simulator.mitm[sysid] = {"strategy": "blackout"}

simulator.show()

In [ ]:
orac = simulator.launch()
orac.run()

## What to observe

- The drone flies the **full north mission** and does **not** turn east.
- `simulator/logs/mitm/mitm_1.log` — `strategy=BlackoutStrategy`.
- `simulator/logs/GCSs/GCS_BLUE_*.log` — **no** `GCS intervention` line (the
  GCS never saw `MISSION_CURRENT` reach its trigger).
- The run still terminates cleanly because `HEARTBEAT` and `LOGIC_DONE` are
  allowed through.